# Test 1: does the model predict, or remember?

The plan was committed in `24ba7c3`, and the switch to Gemma in `5e1a7e6`, both before any of these answers existed: see [PLAN.md](PLAN.md).

- **Model:** `gemma-3-4b-it`, 4-bit, run on this laptop with llama.cpp. Its training data ends in Aug 2024.
- **Three ways of asking:** *named* (index name, dates, 20 days of prices), *hidden* (the same prices with name and dates removed), *date only* (no prices).
- **Two periods:** *before* (Jan 2017 – Aug 2024) and *after* (Sep 2024 – Oct 2026), 300 random days per market in each.
- **Main score:** AUC. 0.50 is a coin flip.

This notebook only reads the saved answers. `run_accuracy.py` collects them.

In [1]:
import sys
sys.path.insert(0, "..")  # the main project's code is one folder up

import json

import pandas as pd
from sklearn.metrics import roc_auc_score

from data import MARKETS, clean, load_market
from evaluation import bootstrap_range, holm, majority_direction, mcnemar
from llm_harness.answers import last_word, up_probability
from llm_harness.client import SAVED
from make_fake_data import fake_prices

RUN = "accuracy_gemma"
PERIODS = ["before", "after"]
MODES = ["hidden", "named", "date only"]

prices = {market: load_market(market, include_holdout=True) for market in MARKETS}
prices["fake"] = clean(fake_prices(seed=99), "fake")

## The answers

One row per question: what the model said, what happened, and what the fair bar would have said.

In [2]:
def fair_calls(df):
    """For each year, the direction that was more common in all earlier days."""
    up = df["ret"].dropna() > 0
    return {year: majority_direction(up[up.index.year < year]) for year in range(2012, 2027)}

fair = {market: fair_calls(df) for market, df in prices.items()}

rows = []
for line in (SAVED / f"{RUN}.jsonl").read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    day, market, reply = pd.Timestamp(record["day"]), record["market"], record["reply"]
    thinking = record["mode"].endswith("thinking")
    rows.append({
        "day": day, "market": market, "period": record["period"], "mode": record["mode"],
        "word": last_word(reply) if thinking else (reply["choices"][0]["message"]["content"] or "").strip().lower(),
        "p_up": None if thinking else up_probability(reply),
        "up": bool(prices[market].loc[day, "ret"] > 0),
        "fair_call": fair[market][day.year],
    })
answers = pd.DataFrame(rows).set_index("day").sort_index()

unclear = ~answers["word"].isin(["up", "down"])
print(f"{len(answers)} answers, {unclear.sum()} of them neither up nor down (left out)")
answers = answers[~unclear]
answers["call_up"] = answers["word"] == "up"
real = answers[answers["market"] != "fake"]

11100 answers, 0 of them neither up nor down (left out)


## Overview

Every way of asking in every period, six markets together. **p vs fair** is McNemar's paired test of accuracy against the fair bar.

In [3]:
def overview(group):
    has_probabilities = group["p_up"].notna().all()
    return pd.Series({
        "questions": len(group),
        "said up %": 100 * group["call_up"].mean(),
        "days up %": 100 * group["up"].mean(),
        "accuracy %": 100 * (group["call_up"] == group["up"]).mean(),
        "fair bar %": 100 * (group["fair_call"] == group["up"]).mean(),
        "p vs fair": mcnemar(group["up"], group["call_up"], group["fair_call"])[2],
        "AUC": roc_auc_score(group["up"], group["p_up"].astype(float)) if has_probabilities else float("nan"),
    })

table = real.groupby(["period", "mode"])[real.columns].apply(overview)
table.loc[[(period, mode) for period in PERIODS for mode in MODES]].round(3)

questions  said up %  days up %  accuracy %  fair bar %  \
period mode                                                                 
before hidden        1800.0     99.556     51.833      51.722      51.833   
       named         1800.0     99.556     51.833      51.833      51.833   
       date only     1800.0    100.000     51.833      51.833      51.833   
after  hidden        1800.0     99.167     51.278      51.333      51.278   
       named         1800.0     99.389     51.278      51.000      51.278   
       date only     1800.0    100.000     51.278      51.278      51.278   

                  p vs fair    AUC  
period mode                         
before hidden         0.727  0.508  
       named          1.000  0.503  
       date only      1.000  0.498  
after  hidden         1.000  0.494  
       named          0.227  0.490  
       date only      1.000  0.513

## The three planned comparisons

AUC with its 95% range (whole weeks resampled 2,000 times).

1. **Hidden:** can it predict from prices alone?
2. **Named minus hidden,** on the same days: do the name and dates add anything?
3. **Date only:** pure memory.

Memory would show as comparison 2 or 3 above its coin-flip value *before* the cutoff and not *after*.

In [4]:
def auc(frame, column="p_up"):
    return roc_auc_score(frame["up"], frame[column].astype(float))

def with_range(daily, statistic):
    """The number itself and its 95% range."""
    low, high = bootstrap_range(daily, statistic)
    return {"value": statistic(daily), "low": low, "high": high}

def one(mode, period):
    return real[(real["mode"] == mode) & (real["period"] == period)]

def side_by_side(first, second, period):
    """Two ways of asking, matched on the same market and day."""
    left = one(first, period).set_index("market", append=True)
    right = one(second, period).set_index("market", append=True)[["p_up", "call_up"]].add_suffix("_other")
    return left.join(right, how="inner").reset_index("market")

comparisons = {}
for period in PERIODS:
    comparisons["1. hidden (coin flip = 0.50)", period] = with_range(one("hidden", period), auc)
    comparisons["2. named minus hidden (no effect = 0)", period] = with_range(
        side_by_side("named", "hidden", period), lambda both: auc(both) - auc(both, "p_up_other"))
    comparisons["3. date only (coin flip = 0.50)", period] = with_range(one("date only", period), auc)
comparisons = pd.DataFrame(comparisons).T
comparisons.round(3)

,,value,low,high
1. hidden (coin flip = 0.50),before,0.508,0.483,0.532
2. named minus hidden (no effect = 0),before,-0.004,-0.031,0.022
3. date only (coin flip = 0.50),before,0.498,0.472,0.525
1. hidden (coin flip = 0.50),after,0.494,0.467,0.521
2. named minus hidden (no effect = 0),after,-0.004,-0.034,0.026
3. date only (coin flip = 0.50),after,0.513,0.488,0.540


The same three comparisons on accuracy, with paired tests. **After adjusting** uses Holm's method within each comparison.

In [5]:
tests = {}
for period in PERIODS:
    hidden, date_only, both = one("hidden", period), one("date only", period), side_by_side("named", "hidden", period)
    tests["1. hidden vs fair bar", period] = mcnemar(hidden["up"], hidden["call_up"], hidden["fair_call"])
    tests["2. named vs hidden", period] = mcnemar(both["up"], both["call_up"], both["call_up_other"])
    tests["3. date only vs fair bar", period] = mcnemar(date_only["up"], date_only["call_up"], date_only["fair_call"])
tests = pd.DataFrame(tests, index=["days only the first got right", "days only the second got right", "p"]).T
tests["p after adjusting"] = tests["p"].groupby(level=0, group_keys=False).apply(holm)
tests.round(3)

,,days only the first got right,days only the second got right,p,p after adjusting
1. hidden vs fair bar,before,3.0,5.0,0.727,1.000
2. named vs hidden,before,9.0,7.0,0.804,0.804
3. date only vs fair bar,before,0.0,0.0,1.000,1.000
1. hidden vs fair bar,after,8.0,7.0,1.000,1.000
2. named vs hidden,after,10.0,16.0,0.327,0.654
3. date only vs fair bar,after,0.0,0.0,1.000,1.000


## Check on the harness: fake data

Made-up random prices, asked the hidden way. Nothing can be predicted, so AUC should be about 0.50.

In [6]:
fake = answers[answers["market"] == "fake"]
display(overview(fake).round(3).to_frame("fake data").T)
pd.Series(with_range(fake, auc), name="AUC on fake data").round(3).to_frame().T

,questions,said up %,days up %,accuracy %,fair bar %,p vs fair,AUC
fake data,300.0,100.0,50.667,50.667,49.333,0.708,0.522


,value,low,high
AUC on fake data,0.522,0.456,0.584


## A closer look by year (not in the plan)

This table wasn't in the plan, but it was written before the answers were seen. The plan expected memory to show in famous stretches such as March 2020, so this shows AUC by year for the two ways of asking that could use memory.

In [7]:
by_year = {}
for mode in ["date only", "named", "hidden"]:
    group = real[real["mode"] == mode]
    by_year[mode] = {year: auc(in_year) for year, in_year in group.groupby(group.index.year) if in_year["up"].nunique() == 2}
by_year = pd.DataFrame(by_year)
by_year["questions"] = real[real["mode"] == "hidden"].groupby(lambda day: day.year).size()
by_year.round(3)

,date only,named,hidden,questions
2017,0.439,0.557,0.447,224
2018,0.521,0.538,0.498,231
2019,0.531,0.427,0.482,226
2020,0.511,0.531,0.560,230
2021,0.514,0.482,0.490,242
2022,0.486,0.504,0.504,247
2023,0.443,0.472,0.547,242
2024,0.520,0.511,0.507,440
2025,0.521,0.476,0.508,848
2026,0.525,0.504,0.482,670


## What this shows

Everything came out as the plan expected: no skill, and no sign of memory.

- **Gemma says "up" to almost everything:** 99–100% of answers in every way of asking. Its accuracy is the fair bar's, because the fair bar was "up" in nearly every year.
- **Prices alone: no skill.** Hidden AUC is 0.508 before its cutoff (range 0.483 to 0.532) and 0.494 after.
- **The name and dates add nothing:** named minus hidden is −0.004 in both periods.
- **No memory:** with only a date, AUC is 0.498 for days it could have read about and 0.513 for days it couldn't. Memory would show the opposite.
- **The harness passes its check:** on fake data AUC is 0.522, with a range (0.456 to 0.584) that includes 0.50.
- **The year-by-year table** moves between 0.43 and 0.56, about what 220–850 days of chance gives. No year stands out across all three ways of asking.

**The limit:** a 4B model holds little detail, so "no memory" may say more about its size than about memory in general. Whether a bigger model remembers is still open: the Nemotron run stopped at 829 answers.